# Bronze | Helper de ingestão batch

Função compartilhada para ingestão das fontes da Base dos Dados.
Cada execução gera um lote_id único e adiciona o snapshot em modo APPEND.
Regras:
- Bronze é histórica e append-only;
- cada carga recebe lote_id;
- cada carga recebe data_ingestao;
- cada carga recebe fonte;
- carga vazia é bloqueada;
- metadados obrigatórios nulos são bloqueados;
- falhas são persistidas no monitoramento e propagadas com raise.

In [0]:


import time
import uuid

from datetime import datetime, timezone

from google.cloud import bigquery
from google.oauth2 import service_account

from pyspark.sql.functions import current_timestamp, lit


import json
import os


def _config(nome, padrao=None, obrigatorio=False):
    """Lê configuração de variável de ambiente (ou spark.conf 'tc.<nome>'), nunca do código."""
    valor = os.environ.get(nome)
    if not valor:
        try:
            valor = spark.conf.get(f"tc.{nome.lower()}", None)
        except Exception:
            valor = None
    if not valor:  # serverless: não há env var/conf de cluster -> usar Databricks Secrets (escopo tech-challenge, chave = nome em minúsculas, com '-')
        try:
            valor = dbutils.secrets.get("tech-challenge", nome.lower().replace("_", "-"))
        except Exception:
            valor = None
    valor = valor or padrao
    if obrigatorio and not valor:
        raise RuntimeError(
            f"Configuração ausente: defina a variável de ambiente {nome} "
            f"ou spark.conf 'tc.{nome.lower()}' (ver README, seção 'Como executar')."
        )
    return valor


# Projeto GCP que paga as consultas ao BigQuery (obrigatório, sem valor padrão).
GCP_PROJECT = _config("GCP_PROJECT", obrigatorio=True)

# Credencial: preferir Databricks Secrets (escopo/chave abaixo). Alternativa: arquivo JSON
# fora do repositório, apontado por GCP_CREDENTIALS_PATH.
SECRET_SCOPE = _config("TC_SECRET_SCOPE", "tech-challenge")
SECRET_KEY = _config("TC_SECRET_KEY", "gcp-sa-json")
GCP_CREDENTIALS_PATH = _config("GCP_CREDENTIALS_PATH")


def _credenciais_gcp():
    try:
        info = json.loads(dbutils.secrets.get(SECRET_SCOPE, SECRET_KEY))
        return service_account.Credentials.from_service_account_info(
            info, scopes=["https://www.googleapis.com/auth/cloud-platform"])
    except Exception as exc_secret:
        if GCP_CREDENTIALS_PATH:
            return service_account.Credentials.from_service_account_file(GCP_CREDENTIALS_PATH)
        raise RuntimeError(
            f"Sem credencial GCP: secret {SECRET_SCOPE}/{SECRET_KEY} indisponível "
            f"({type(exc_secret).__name__}) e GCP_CREDENTIALS_PATH não definido."
        )


In [0]:
def _monitoring_tables():

    spark.sql("""
        CREATE SCHEMA IF NOT EXISTS workspace.monitoring
    """)

    spark.sql("""
        CREATE TABLE IF NOT EXISTS workspace.monitoring.pipeline_runs (
            run_id STRING,
            layer STRING,
            entity STRING,
            started_at TIMESTAMP,
            finished_at TIMESTAMP,
            duration_seconds DOUBLE,
            status STRING,
            row_count BIGINT,
            error_message STRING
        )
        USING DELTA
    """)

    spark.sql("""
        CREATE TABLE IF NOT EXISTS workspace.monitoring.finops_ingestion (
            run_id STRING,
            entity STRING,
            source_table STRING,
            bytes_processed BIGINT,
            bytes_billed BIGINT,
            row_count BIGINT,
            recorded_at TIMESTAMP
        )
        USING DELTA
    """)


In [0]:
def ingest_bq_table(
    entity,
    source_table,
    target_table,
    max_bytes_billed=20_000_000_000,
    manter_ultimos_lotes=None
):

    """
    Consulta um snapshot completo da fonte BigQuery
    e adiciona o lote imutável na Bronze.
    """

    _monitoring_tables()

    run_id = str(uuid.uuid4())

    started = datetime.now(timezone.utc).replace(tzinfo=None)

    tick = time.perf_counter()

    row_count = 0

    try:

        # ---------------------------------------------------------
        # 1. Autenticação
        # ---------------------------------------------------------

        credentials = _credenciais_gcp()

        client = bigquery.Client(
            credentials=credentials,
            project=GCP_PROJECT
        )


        # ---------------------------------------------------------
        # 2. Consulta da fonte
        # ---------------------------------------------------------

        job_config = bigquery.QueryJobConfig(
            maximum_bytes_billed=int(max_bytes_billed)
        )

        query_job = client.query(
            f"SELECT * FROM `{source_table}`",
            job_config=job_config
        )

        pdf = query_job.to_dataframe(
            create_bqstorage_client=True
        )

        row_count = len(pdf)

        # Estatísticas do job só são confiáveis após recarregar o job; se ainda
        # vierem zeradas, estima os bytes com um dry run (não gera cobrança).
        query_job.reload()
        bytes_proc = int(query_job.total_bytes_processed or 0)
        bytes_bill = int(query_job.total_bytes_billed or 0)
        if bytes_proc == 0:
            dry = client.query(
                f"SELECT * FROM `{source_table}`",
                job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False),
            )
            bytes_proc = int(dry.total_bytes_processed or 0)
            if bytes_bill == 0:
                # cobrança mínima de 10 MiB por consulta, arredondada para cima
                bytes_bill = max(bytes_proc, 10 * 1024 * 1024)


        # ---------------------------------------------------------
        # 3. Bloqueio de carga vazia
        # ---------------------------------------------------------

        if row_count == 0:

            raise ValueError(
                f"A fonte retornou 0 registros: {source_table}"
            )


        # ---------------------------------------------------------
        # 4. Criação do DataFrame Bronze
        # ---------------------------------------------------------

        sdf = (
            spark
            .createDataFrame(pdf)
            .withColumn(
                "data_ingestao",
                current_timestamp()
            )
            .withColumn(
                "fonte",
                lit("Base dos Dados - BigQuery")
            )
            .withColumn(
                "lote_id",
                lit(run_id)
            )
        )


        # ---------------------------------------------------------
        # 5. Validação bloqueante dos metadados
        # ---------------------------------------------------------

        registros_invalidos = (
            sdf
            .filter(
                """
                lote_id IS NULL
                OR data_ingestao IS NULL
                OR fonte IS NULL
                """
            )
            .limit(1)
            .count()
        )

        if registros_invalidos > 0:

            raise ValueError(
                f"Metadados obrigatórios nulos antes da gravação: "
                f"{target_table}"
            )


        # ---------------------------------------------------------
        # 6. Gravação histórica na Bronze
        # ---------------------------------------------------------

        (
            sdf
            .write
            .format("delta")
            .mode("append")
            .option("mergeSchema", "true")
            .saveAsTable(target_table)
        )


        # ---------------------------------------------------------
        # 6.1 Retenção opcional (FinOps). Padrão: None = preserva TODO o histórico.
        # Só use se a política de retenção estiver documentada no README.
        # ---------------------------------------------------------

        if manter_ultimos_lotes:
            spark.sql(f"""
                DELETE FROM {target_table}
                WHERE lote_id NOT IN (
                    SELECT lote_id FROM (
                        SELECT lote_id, MAX(data_ingestao) AS ult
                        FROM {target_table}
                        GROUP BY lote_id
                        ORDER BY ult DESC
                        LIMIT {int(manter_ultimos_lotes)}
                    )
                )
            """)

        # ---------------------------------------------------------
        # 7. Finalização
        # ---------------------------------------------------------

        finished = datetime.now(timezone.utc).replace(tzinfo=None)

        duration = time.perf_counter() - tick


        # ---------------------------------------------------------
        # 8. Monitoramento
        # ---------------------------------------------------------

        spark.createDataFrame(
            [
                (
                    run_id,
                    "BRONZE",
                    entity,
                    started,
                    finished,
                    duration,
                    "SUCESSO",
                    row_count,
                    None
                )
            ],
            """
            run_id string,
            layer string,
            entity string,
            started_at timestamp,
            finished_at timestamp,
            duration_seconds double,
            status string,
            row_count long,
            error_message string
            """
        ).write \
         .format("delta") \
         .mode("append") \
         .saveAsTable(
             "workspace.monitoring.pipeline_runs"
         )


        # ---------------------------------------------------------
        # 9. FinOps
        # ---------------------------------------------------------

        spark.createDataFrame(
            [
                (
                    run_id,
                    entity,
                    source_table,
                    bytes_proc,
                    bytes_bill,
                    row_count,
                    finished
                )
            ],
            """
            run_id string,
            entity string,
            source_table string,
            bytes_processed long,
            bytes_billed long,
            row_count long,
            recorded_at timestamp
            """
        ).write \
         .format("delta") \
         .mode("append") \
         .saveAsTable(
             "workspace.monitoring.finops_ingestion"
         )


        # ---------------------------------------------------------
        # 10. Evidência visual
        # ---------------------------------------------------------

        print("=" * 60)
        print("✅ INGESTÃO CONCLUÍDA COM SUCESSO")
        print(f"📌 Tabela destino: {target_table}")
        print(f"📊 Registros ingeridos: {row_count}")
        print(f"📦 Lote: {run_id}")
        print(f"⏱️ Duração: {duration:.1f}s")
        print("=" * 60)


        return run_id


    except Exception as exc:

        finished = datetime.now(timezone.utc).replace(tzinfo=None)

        duration = time.perf_counter() - tick


        # Registra a falha
        spark.createDataFrame(
            [
                (
                    run_id,
                    "BRONZE",
                    entity,
                    started,
                    finished,
                    duration,
                    "FALHA",
                    row_count,
                    str(exc)[:2000]
                )
            ],
            """
            run_id string,
            layer string,
            entity string,
            started_at timestamp,
            finished_at timestamp,
            duration_seconds double,
            status string,
            row_count long,
            error_message string
            """
        ).write \
         .format("delta") \
         .mode("append") \
         .saveAsTable(
             "workspace.monitoring.pipeline_runs"
         )


        # IMPORTANTE:
        # não engole o erro.
        raise